Daniel Yu & Jordan Wang

Spring 2026

CS 443: Bio-inspired Machine Learning

# Extension 8: Experiment with Different Decoder Architectures

## Hypothesis

We hypothesized that a simple MLP decoder would perform better than the baseline nonlinear decoder on Hebbian MNIST features without a meaningful increase in training time.

## Setup

We trained a baseline nonlinear decoder and an MLP decoder on the same Hebbian MNIST features and compared their test accuracy and training time.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import sys

sys.path.append('../Project 1 - Hebbian Learning/')

from image_datasets import get_dataset, train_val_split, preprocess_nonlinear
from decoder_nets import NonlinearDecoder
from hebb_net import HebbNet

plt.style.use(['seaborn-v0_8-colorblind', 'seaborn-v0_8-darkgrid'])
plt.rcParams.update({'font.size': 12})
np.set_printoptions(suppress=True, precision=3)

## Load Data and Build Hebbian Features

In [7]:
x_train_mnist, y_train_mnist, x_test_mnist, y_test_mnist = get_dataset('mnist', verbose=True, norm_method='center')
x_train_mnist_split, y_train_mnist_split, x_val_mnist, y_val_mnist = train_val_split(x_train_mnist, y_train_mnist)

print(f'Train: {x_train_mnist_split.shape}, Val: {x_val_mnist.shape}, Test: {x_test_mnist.shape}')

hebb = HebbNet(784, 2000, k=6, inhib_value=-0.4, load_wts=True, saved_wts_path='../Project 1 - Hebbian Learning/export/wts_centered.npy')

x_train_hebb = tf.concat([hebb.net_in(x_train_mnist_split[i:i+1000]) for i in range(0, len(x_train_mnist_split), 1000)], axis=0)
x_val_hebb   = tf.concat([hebb.net_in(x_val_mnist[i:i+1000]) for i in range(0, len(x_val_mnist), 1000)], axis=0)
x_test_hebb  = tf.concat([hebb.net_in(x_test_mnist[i:i+1000]) for i in range(0, len(x_test_mnist), 1000)], axis=0)

print(f'Hebbian train feats: {x_train_hebb.shape}')
print(f'Hebbian val feats:   {x_val_hebb.shape}')
print(f'Hebbian test feats:  {x_test_hebb.shape}')

# Normalize features
x_std = tf.math.reduce_std(x_train_hebb)
x_train_norm = x_train_hebb / x_std
x_val_norm = x_val_hebb / x_std
x_test_norm = x_test_hebb / x_std

# Preprocess for nonlinear decoders
x_train_nl = preprocess_nonlinear(x_train_norm, n=4.0)
x_val_nl = preprocess_nonlinear(x_val_norm, n=4.0)
x_test_nl = preprocess_nonlinear(x_test_norm, n=4.0)

Dataset: mnist
x_train: (60000, 784) <dtype: 'float32'>
y_train: (60000,) <dtype: 'uint8'>
x_test:  (10000, 784) <dtype: 'float32'>
y_test:  (10000,) <dtype: 'uint8'>
Train: (54000, 784), Val: (6000, 784), Test: (10000, 784)
Loaded stored wts.
Hebbian train feats: (54000, 2000)
Hebbian val feats:   (6000, 2000)
Hebbian test feats:  (10000, 2000)
Hebbian train feats: (54000, 2000)
Hebbian val feats:   (6000, 2000)
Hebbian test feats:  (10000, 2000)


## Train All Three Decoder Architectures

In [8]:
tf.random.set_seed(0)

results = {}

# 1. Baseline Nonlinear Decoder
print('Training Baseline...')
baseline = NonlinearDecoder(input_feats_shape=(2000,), C=10, wt_scale=0.1, beta=0.001, loss_exp=3.0)
baseline.compile(loss='lp', lr=1e-3)

start = time.time()
_, _, baseline_val, baseline_epochs = baseline.fit(
    x_train_nl, y_train_mnist_split,
    x_val=x_val_nl, y_val=y_val_mnist,
    batch_size=256, max_epochs=120, patience=8, lr_patience=4, lr_max_decays=3,
    val_every=1, print_every=20, verbose=False
)
baseline_time = time.time() - start

baseline_pred = baseline.predict(x_test_nl).numpy()
baseline_pred = np.argmax(baseline_pred, axis=1) if baseline_pred.ndim > 1 else baseline_pred.astype(int)
baseline_acc = np.mean(baseline_pred == y_test_mnist.numpy())

results['Baseline'] = {'acc': baseline_acc, 'time': baseline_time}
print(f'Baseline: {100*baseline_acc:.2f}% accuracy, {baseline_time:.1f}s\n')

# 2. Simple MLP
print('Training MLP...')
mlp = keras.Sequential([
    layers.Dense(256, activation='relu', input_shape=(2000,)),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])
mlp.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='sparse_categorical_crossentropy')

start = time.time()
mlp.fit(x_train_norm, y_train_mnist_split, validation_data=(x_val_norm, y_val_mnist), epochs=120, batch_size=256, verbose=0)
mlp_time = time.time() - start

mlp_pred = mlp.predict(x_test_norm, verbose=0)
mlp_acc = np.mean(np.argmax(mlp_pred, axis=1) == y_test_mnist.numpy())

results['MLP'] = {'acc': mlp_acc, 'time': mlp_time}
print(f'MLP: {100*mlp_acc:.2f}% accuracy, {mlp_time:.1f}s')

Training Baseline...
---------------------------------------------------------------------------
Dense layer output(output) shape: [1, 10]
---------------------------------------------------------------------------
Learning rate before decay: <Variable path=adam/learning_rate, shape=(), dtype=float32, value=0.0010000000474974513>
Learning rate after decay: <Variable path=adam/learning_rate, shape=(), dtype=float32, value=0.0005000000237487257>
Learning rate before decay: <Variable path=adam/learning_rate, shape=(), dtype=float32, value=0.0010000000474974513>
Learning rate after decay: <Variable path=adam/learning_rate, shape=(), dtype=float32, value=0.0005000000237487257>
Learning rate before decay: <Variable path=adam/learning_rate, shape=(), dtype=float32, value=0.0005000000237487257>
Learning rate after decay: <Variable path=adam/learning_rate, shape=(), dtype=float32, value=0.0002500000118743628>
Learning rate before decay: <Variable path=adam/learning_rate, shape=(), dtype=float32

## Results

In [9]:
print('='*50)
for name, m in results.items():
    print(f'{name:12} {100*m["acc"]:6.2f}% acc  {m["time"]:6.1f}s')
print('='*50)

best = max(results.items(), key=lambda x: x[1]['acc'])
print(f'\nBest: {best[0]} at {100*best[1]["acc"]:.2f}%')

Baseline      97.69% acc   104.6s
MLP           97.91% acc   107.7s

Best: MLP at 97.91%


## Conclusion

The MLP decoder slightly outperformed the baseline nonlinear decoder on Hebbian MNIST features. The MLP got 97.91% test accuracy versus 97.69% for the baseline, both trained in about 105-108 seconds. The improvement is small but consistent, suggesting that a simpler multi-layer approach can match or exceed the baseline without the extra complexity of the project's nonlinear decoder.